In [ ]:
operation = "blocked"
probe_id = ""
runtime_evidence_json = "{}"
run_config_json = "{}"
selected_demos_json = "[]"
native_profile_json = "{}"


In [ ]:
import base64
import json
import re
import uuid
from datetime import datetime, timezone
spark.conf.set("spark.fabric.session.highConcurrency.enabled", "true")
if operation != "run-native":
    config = json.loads(run_config_json)
    selected = json.loads(selected_demos_json)
    runtime_request = json.loads(runtime_evidence_json)
if operation == "preflight" and runtime_request:
    import hashlib
    from datetime import timedelta
    request = runtime_request
    if not isinstance(request, dict) or set(request) != {"nonce", "run_id", "tenant_id", "identity_id", "producer", "demo-2"}:
        raise ValueError("Invalid runtime evidence request")
    if not isinstance(probe_id, str) or not re.fullmatch(r"[a-z0-9][a-z0-9-]{0,63}", probe_id):
        raise ValueError("A safe, unique preflight probe_id is required")
    if (not isinstance(request["nonce"], str) or not re.fullmatch(r"[A-Za-z0-9_-]{16,128}", request["nonce"])
            or not isinstance(request["run_id"], str) or not re.fullmatch(r"[A-Za-z0-9_-]{1,128}", request["run_id"])):
        raise ValueError("Invalid runtime nonce or run ID")
    if (not isinstance(selected, list) or any(not isinstance(item, str) or item not in ("demo-1", "demo-2", "demo-3") for item in selected)
            or len(selected) != len(set(selected)) or "demo-2" not in selected):
        raise ValueError("Select distinct demos including demo-2")
    selected = sorted(selected)
    def guid(value):
        if not isinstance(value, str):
            raise ValueError("Expected a UUID")
        return str(uuid.UUID(value))
    producer = request["producer"]
    target = request["demo-2"]
    if (not isinstance(producer, dict) or set(producer) != {"workspace_id", "staging_lakehouse_id", "notebook_id", "definition_sha256", "environment_id"}
            or not isinstance(target, dict) or set(target) != {"workspace_id", "item_id", "database", "fact", "mapping", "query_service_uri", "ingestion_service_uri"}):
        raise ValueError("Invalid producer or destination binding")
    tenant_id, identity_id = guid(request["tenant_id"]), guid(request["identity_id"])
    producer = dict(producer)
    target = dict(target)
    for key in ("workspace_id", "staging_lakehouse_id", "notebook_id", "environment_id"):
        producer[key] = guid(producer[key])
    for key in ("workspace_id", "item_id"):
        target[key] = guid(target[key])
    if not isinstance(producer["definition_sha256"], str) or not re.fullmatch(r"[0-9a-f]{64}", producer["definition_sha256"]):
        raise ValueError("Invalid definition digest")
    synthetic_target = bool(re.fullmatch(r"demo2synthetic_[0-9a-f]{12}", request["run_id"]))
    expected_fact = "SyntheticFactSales" if synthetic_target else "FactSales"
    if (target["fact"] != expected_fact or target["mapping"] != "SalesCsvV1"
            or not isinstance(target["database"], str) or not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]{0,127}", target["database"])):
        raise ValueError("Invalid Eventhouse database, table or mapping")
    from fabric_runtime import _kusto_endpoint
    target["query_service_uri"] = _kusto_endpoint(target["query_service_uri"])
    target["ingestion_service_uri"] = _kusto_endpoint(target["ingestion_service_uri"])
    context = notebookutils.runtime.context
    fingerprint = hashlib.sha256(json.dumps(config, sort_keys=True, separators=(",", ":"), ensure_ascii=False, allow_nan=False).encode("utf-8")).hexdigest()
    checks = []
    def check(capability, resource, method, result):
        checks.append({"capability": capability, "resource": resource, "method": method, "result": result})
    evidence = {"version": 1, "kind": "producer-runtime-preflight", "nonce": request["nonce"], "probe_id": probe_id, "run_id": request["run_id"], "tenant_id": tenant_id, "identity_id": identity_id, "effective_config_sha256": fingerprint, "selected_demos": selected, "producer": producer, "targets": {"demo-2": dict(target, checks=checks)}, "status": "blocked", "authority": "runtime-observation-only"}
    # These are caller assertions, not proof of the deployed definition or environment.
    evidence["producer"]["job_id"] = None
    try:
        observed_run_id = guid(context.get("currentRunId"))
    except (ValueError, TypeError):
        observed_run_id = None
    try:
        observed_notebook_id = guid(context.get("currentNotebookId"))
    except (ValueError, TypeError):
        observed_notebook_id = None
    evidence["producer"]["observed_run_id"] = observed_run_id
    evidence["producer"]["observed_notebook_id"] = observed_notebook_id
    context_checks = {"pipeline": context.get("isForPipeline") is True, "workspace": context.get("currentWorkspaceId") == producer["workspace_id"], "lakehouse": context.get("defaultLakehouseId") == producer["staging_lakehouse_id"], "notebook": observed_notebook_id == producer["notebook_id"]}
    evidence["producer"]["runtime_context_checks"] = context_checks
    context_matches = all(context_checks.values())
    check("runtime-binding", producer["notebook_id"], "notebook-runtime-context", "passed" if context_matches else "failed")
    identities = {}
    for audience in ("pbi", "storage", "kusto"):
        try:
            token = notebookutils.credentials.getToken(audience)
            payload = token.split(".")[1]
            claims = json.loads(base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)))
            oid, tid = guid(claims["oid"]), guid(claims["tid"])
            identities[audience] = {"identity_id": oid, "tenant_id": tid}
            check("token-identity", audience, "runtime-token-claims", "passed" if (oid, tid) == (identity_id, tenant_id) else "failed")
        except Exception:
            check("token-identity", audience, "runtime-token-claims", "failed")
    evidence["token_identities"] = identities
    prerequisites = context_matches and len(identities) == 3 and all(value == {"identity_id": identity_id, "tenant_id": tenant_id} for value in identities.values())
    try:
        import importlib.metadata
        import azure.identity
        import azure.kusto.ingest
        import eventhouse_core
        import ingestion
        if not callable(getattr(ingestion.EventhouseAdapter, "ingestion_status", None)):
            raise RuntimeError("Terminal ingestion status adapter unavailable")
        versions = {name: importlib.metadata.version(name) for name in ("azure-identity", "azure-kusto-data", "azure-kusto-ingest")}
        if any(not isinstance(value, str) or not re.fullmatch(r"[0-9][A-Za-z0-9.!+_-]{0,63}", value) for value in versions.values()):
            raise RuntimeError("Invalid package version")
        evidence["runtime_packages"] = versions
        check("runtime-dependencies", producer["environment_id"], "runtime-import-and-version", "passed")
    except Exception:
        check("runtime-dependencies", producer["environment_id"], "runtime-import-and-version", "failed")
    # The names are fixed constants; the database is separately restricted above.
    try:
        if not prerequisites:
            raise RuntimeError("Runtime binding or token identities did not match")
        from azure.kusto.data import KustoClient, KustoConnectionStringBuilder
        connection = KustoConnectionStringBuilder.with_token_provider(target["query_service_uri"], lambda: notebookutils.credentials.getToken("kusto"))
        client = KustoClient(connection)
        response = client.execute_query(target["database"], "print ProducerRuntimeProbe=1")
        rows = list(response.primary_results[0])
        if len(rows) != 1 or rows[0]["ProducerRuntimeProbe"] != 1:
            raise RuntimeError("Unexpected query response")
        check("query", target["query_service_uri"], "kusto-read-only-query", "passed")
    except Exception:
        check("query", target["query_service_uri"], "kusto-read-only-query", "failed")
        client = None
    for capability, command in (("table-schema-read", f".show table {expected_fact} schema as json"), ("mapping-read", f".show table {expected_fact} ingestion csv mappings")):
        try:
            if client is None:
                raise RuntimeError("Query client unavailable")
            response = client.execute_mgmt(target["database"], command)
            if not list(response.primary_results[0]):
                raise RuntimeError("Empty metadata response")
            check(capability, target["fact"], "kusto-read-only-management", "passed")
        except Exception:
            check(capability, target["fact"], "kusto-read-only-management", "failed")
    if synthetic_target:
        check("conditional-storage", producer["staging_lakehouse_id"], "read-only-synthetic-probe", "not-run")
    else:
        try:
            if not prerequisites:
                raise RuntimeError("Runtime binding or token identities did not match")
            from fabric_storage import OneLakeStore, StorageConflict
            store = OneLakeStore(producer["workspace_id"], producer["staging_lakehouse_id"], lambda: notebookutils.credentials.getToken("storage"))
            path = "Files/preflight/runtime-" + probe_id + "-" + uuid.uuid4().hex + ".txt"
            etag = store.create(path, b"runtime-probe")
            if store.read(path) != (b"runtime-probe", etag):
                raise RuntimeError("Storage readback differed")
            try:
                store.create(path, b"duplicate")
            except StorageConflict:
                pass
            else:
                raise RuntimeError("Duplicate create accepted")
            try:
                store.replace(path, b"stale", '"stale-etag"')
            except StorageConflict:
                pass
            else:
                raise RuntimeError("Stale update accepted")
            if store.read(path) != (b"runtime-probe", etag):
                raise RuntimeError("Stale update changed sentinel")
            try:
                store.delete(path, '"stale-etag"')
            except StorageConflict:
                pass
            else:
                raise RuntimeError("Stale delete accepted")
            if store.read(path) != (b"runtime-probe", etag):
                raise RuntimeError("Stale delete changed sentinel")
            new_etag = store.replace(path, b"updated", etag)
            if store.read(path) != (b"updated", new_etag):
                raise RuntimeError("Storage update readback differed")
            store.delete(path, new_etag)
            try:
                store.read(path)
            except FileNotFoundError:
                pass
            else:
                raise RuntimeError("Storage sentinel remains")
            check("conditional-storage", producer["staging_lakehouse_id"], "owned-etag-sentinel", "passed")
        except Exception:
            check("conditional-storage", producer["staging_lakehouse_id"], "owned-etag-sentinel", "failed")
    try:
        if not prerequisites:
            raise RuntimeError("Runtime binding or token identities did not match")
        from azure.kusto.data import KustoConnectionStringBuilder
        from azure.kusto.ingest import QueuedIngestClient
        from synthetic_fixture import peek_status_queues
        ingest = QueuedIngestClient(KustoConnectionStringBuilder.with_token_provider(target["ingestion_service_uri"], lambda: notebookutils.credentials.getToken("kusto")))
        try:
            evidence["status_queue_peeks"] = peek_status_queues(ingest)
        finally:
            ingest.close()
        check("status-queue", target["ingestion_service_uri"], "sdk-success-and-failure-nondestructive-peek", "passed")
    except Exception as exc:
        evidence["status_queue_error_type"] = type(exc).__name__
        check("status-queue", target["ingestion_service_uri"], "sdk-success-and-failure-nondestructive-peek", "failed")
    check("effective-ingestion-authorization", target["item_id"], "authoritative-role-or-approved-ingestion-probe-required", "unverified")
    now = datetime.now(timezone.utc)
    evidence["issued_at_utc"] = now.isoformat()
    evidence["expires_at_utc"] = (now + timedelta(minutes=5)).isoformat()
    # A caller must bind this exact exit value to a freshly observed job and published definition.
    notebookutils.notebook.exit(json.dumps(evidence, sort_keys=True, separators=(",", ":")))
elif operation == "preflight":
    if not isinstance(probe_id, str) or not re.fullmatch(r"[a-z0-9][a-z0-9-]{0,63}", probe_id):
        raise ValueError("A safe, unique preflight probe_id is required")
    context = notebookutils.runtime.context
    def token_identity(audience):
        token = notebookutils.credentials.getToken(audience)
        payload = token.split(".")[1]
        claims = json.loads(base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)))
        if not claims.get("oid") or not claims.get("tid"):
            raise RuntimeError("Token lacks identity claims")
        return claims["oid"], claims["tid"]
    storage_oid, storage_tenant = token_identity("storage")
    evidence = {"probe_id": probe_id, "operation": operation, "parameter_marker": config.get("probe_marker"), "selected_demos": selected, "is_for_pipeline": context["isForPipeline"], "workspace_id": context["currentWorkspaceId"], "lakehouse_id": context["defaultLakehouseId"], "storage_principal_oid": storage_oid, "tenant_id": storage_tenant}
    failures = []
    if config.get("probe_kusto"):
        try:
            kusto_oid, kusto_tenant = token_identity("kusto")
            evidence.update(kusto_principal_oid=kusto_oid, kusto_tenant_id=kusto_tenant)
        except Exception as exc:
            evidence["kusto_token_error"] = type(exc).__name__
            failures.append("Kusto token acquisition")
        try:
            import azure.identity
            import azure.kusto.data
            import azure.kusto.ingest
            evidence["kusto_sdk_available"] = True
        except ImportError as exc:
            evidence["kusto_sdk_available"] = False
            evidence["kusto_sdk_error"] = type(exc).__name__
            failures.append("Kusto SDK imports")
    if config.get("probe_bundle"):
        try:
            import fabric_delivery
            import fabric_ledger
            import fabric_runner
            import fabric_runtime
            import fabric_storage
            if "demo-2" in selected:
                import eventhouse_core
                import ingestion
                if not callable(getattr(ingestion.EventhouseAdapter, "ingestion_status", None)):
                    raise ImportError("Eventhouse terminal status adapter is missing")
                evidence["eventhouse_adapter_available"] = True
            evidence["producer_bundle_available"] = True
        except ImportError as exc:
            evidence["producer_bundle_available"] = False
            evidence["producer_bundle_error"] = type(exc).__name__
            failures.append("Producer module imports")
    if config.get("probe_storage"):
        result = {"path": "Files/preflight/storage-" + probe_id + ".txt", "deleted": False, "step": "create"}
        try:
            from fabric_storage import OneLakeStore, StorageConflict
            store = OneLakeStore(context["currentWorkspaceId"], context["defaultLakehouseId"], lambda: notebookutils.credentials.getToken("storage"))
            first_etag = store.create(result["path"], b"before")
            if store.read(result["path"]) != (b"before", first_etag):
                raise RuntimeError("Conditional create readback differed")
            try:
                store.create(result["path"], b"duplicate")
            except StorageConflict:
                result["duplicate_create_rejected"] = True
            else:
                raise RuntimeError("Duplicate create was not rejected")
            try:
                store.replace(result["path"], b"stale", '"stale-etag"')
            except StorageConflict:
                result["stale_update_rejected"] = True
            else:
                raise RuntimeError("Stale update was not rejected")
            new_etag = store.replace(result["path"], b"after", first_etag)
            if store.read(result["path"]) != (b"after", new_etag):
                raise RuntimeError("Conditional update readback differed")
            result["update_readback_verified"] = True
            result["step"] = "stale-delete"
            try:
                store.delete(result["path"], first_etag)
            except StorageConflict:
                result["stale_delete_rejected"] = True
            else:
                raise RuntimeError("Stale delete was not rejected")
            if store.read(result["path"]) != (b"after", new_etag):
                raise RuntimeError("Stale delete changed the storage sentinel")
            result["step"] = "delete"
            store.delete(result["path"], new_etag)
            result["step"] = "confirm-absent"
            try:
                store.read(result["path"])
            except FileNotFoundError:
                result["deleted"] = True
            else:
                raise RuntimeError("Storage sentinel remains")
        except Exception as exc:
            result["error"] = type(exc).__name__
            if type(exc).__name__ in ("StorageError", "StorageConflict", "StorageOutcomeUnknown"):
                result["diagnostic"] = str(exc)
            failures.append("Conditional storage probe")
        evidence["conditional_storage_probe"] = result
    auth = config.get("eventhouse_auth_probe")
    if auth is not None:
        try:
            if not isinstance(auth, dict) or set(auth) != {"query_endpoint", "database"}:
                raise ValueError("Eventhouse auth probe requires endpoint and database")
            from azure.kusto.data import KustoClient, KustoConnectionStringBuilder
            from fabric_runtime import _kusto_endpoint
            endpoint = _kusto_endpoint(auth["query_endpoint"])
            database = auth["database"]
            if not isinstance(database, str) or not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]{0,127}", database):
                raise ValueError("Invalid KQL database name")
            connection = KustoConnectionStringBuilder.with_token_provider(endpoint, lambda: notebookutils.credentials.getToken("kusto"))
            response = KustoClient(connection).execute_query(database, "print ProducerRuntimeProbe=1")
            rows = list(response.primary_results[0])
            if len(rows) != 1 or rows[0]["ProducerRuntimeProbe"] != 1:
                raise RuntimeError("Eventhouse query evidence differed")
            evidence["eventhouse_sdk_query_verified"] = True
        except Exception as exc:
            evidence["eventhouse_sdk_query_verified"] = False
            evidence["eventhouse_sdk_query_error"] = type(exc).__name__
            failures.append("Eventhouse SDK query")
    access = config.get("destination_access_probe")
    if access is not None:
        if not isinstance(access, dict) or set(access) != {"workspace_id", "lakehouse_id"}:
            raise ValueError("Destination access probe requires workspace and Lakehouse IDs")
        workspace_id = str(uuid.UUID(access["workspace_id"]))
        lakehouse_id = str(uuid.UUID(access["lakehouse_id"]))
        probe_dir = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Files/_preflight/producer-access/{probe_id}"
        probe_file = probe_dir + "/identity-probe.txt"
        expected = "producer access probe " + probe_id + "\n"
        result = {"path": probe_file, "started_at_utc": datetime.now(timezone.utc).isoformat(), "readback_verified": False, "sentinel_deleted": False, "directory_deleted": False}
        created_directory = False
        try:
            if notebookutils.fs.exists(probe_dir):
                raise RuntimeError("Destination probe directory already exists")
            if not notebookutils.fs.mkdirs(probe_dir):
                raise RuntimeError("Destination probe directory creation returned false")
            created_directory = True
            if not notebookutils.fs.put(probe_file, expected, False):
                raise RuntimeError("Destination probe write returned false")
            if notebookutils.fs.head(probe_file, len(expected.encode("utf-8")) + 1) != expected:
                raise RuntimeError("Destination probe readback differed")
            result["readback_verified"] = True
        except Exception as exc:
            result["error"] = type(exc).__name__
        finally:
            if created_directory:
                try:
                    if notebookutils.fs.exists(probe_file) and not notebookutils.fs.rm(probe_file, False):
                        raise RuntimeError("Destination probe file cleanup returned false")
                    result["sentinel_deleted"] = not notebookutils.fs.exists(probe_file)
                    if not result["sentinel_deleted"] or not notebookutils.fs.rm(probe_dir, False):
                        raise RuntimeError("Destination probe directory cleanup failed")
                    result["directory_deleted"] = not notebookutils.fs.exists(probe_dir)
                    if not result["directory_deleted"]:
                        raise RuntimeError("Destination probe directory remains")
                except Exception as exc:
                    result["cleanup_error"] = type(exc).__name__
            result["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
        evidence["destination_access_probe"] = result
        if not all(result[key] for key in ("readback_verified", "sentinel_deleted", "directory_deleted")):
            failures.append("Destination access probe")
    directory = "Files/preflight"
    path = directory + "/" + probe_id + ".json"
    if not notebookutils.fs.exists(directory) and not notebookutils.fs.mkdirs(directory):
        raise RuntimeError("Could not create preflight directory")
    if notebookutils.fs.exists(path):
        raise RuntimeError("Preflight evidence already exists; choose a new probe ID")
    serialized = json.dumps(evidence, sort_keys=True)
    if not notebookutils.fs.put(path, serialized, False):
        raise RuntimeError("Could not write preflight evidence")
    if notebookutils.fs.head(path, len(serialized.encode("utf-8")) + 1) != serialized:
        raise RuntimeError("Preflight evidence did not read back")
    print("Preflight evidence persisted:", path)
    if failures:
        raise RuntimeError("Preflight failed: " + ", ".join(failures))
elif operation in ("synthetic-stage", "synthetic-auth", "synthetic-deliver"):
    from generator import strict_json_loads
    from synthetic_fixture import _uuid, authorization_run, prepare_fixture, stage_fixture, verified_authorization
    from fabric_storage import OneLakeStore
    document = strict_json_loads(run_config_json)
    request = strict_json_loads(runtime_evidence_json)
    selected = strict_json_loads(selected_demos_json)
    def token_identity(audience):
        token = notebookutils.credentials.getToken(audience)
        claims = strict_json_loads(base64.urlsafe_b64decode(token.split(".")[1] + "=" * (-len(token.split(".")[1]) % 4)).decode("utf-8"))
        return _uuid(claims["tid"]), _uuid(claims["oid"])
    context = notebookutils.runtime.context
    kind = "demo2-synthetic-" + operation.split("-")[1] + "-v1"
    run, targets, producer = prepare_fixture(document, selected, probe_id, request, context,
                                              {audience: token_identity(audience) for audience in ("storage", "kusto")}, kind=kind)
    store = OneLakeStore(producer["workspace_id"], producer["staging_lakehouse_id"],
                         lambda: notebookutils.credentials.getToken("storage"))
    if operation == "synthetic-stage":
        manifest_path, manifest = stage_fixture(store, run, targets)
        notebookutils.notebook.exit(json.dumps({"kind": kind, "ready": False,
            "run_id": run.run_id, "batch_class": run.batch_class, "manifest_path": manifest_path,
            "batch_ids": [manifest["batches"][str(n)]["batch_id"] for n in (1, 2)],
            "delivery": "pending", "producer_job_id": None}, sort_keys=True))
    else:
        from fabric_runner import run_fabric_batches
        from fabric_runtime import build_adapters
        target = request["kusto"]
        settings = {"demo-2": {"ingestion_endpoint": target["ingestion_service_uri"],
                 "query_endpoint": target["query_service_uri"], "kql_database_name": target["database"],
                 "sales_table": target["fact"], "csv_mapping": target["mapping"]}}
        adapter = build_adapters(settings, ["demo-2"], notebookutils)
        if operation == "synthetic-auth":
            auth = authorization_run(run)
            manifest_path = f"Files/producer/runs/{auth.run_id}/manifest.json"
            try:
                store.read(manifest_path)
            except FileNotFoundError:
                pass
            else:
                raise RuntimeError("Authorization probe was already initialized; reconcile, never resubmit")
            result = run_fabric_batches(store, auth, adapter, 300, 1, selected_targets=targets)
        else:
            source_id = verified_authorization(store, run, targets)
            manifest_path = f"Files/producer/runs/{run.run_id}/manifest.json"
            result = run_fabric_batches(store, run, adapter, 300, 1, selected_targets=targets,
                                        allow_staged_pending=True)
        manifest = strict_json_loads(store.read(manifest_path)[0].decode("utf-8"))
        if not result["verified"]:
            raise RuntimeError("Synthetic ingestion not terminally verified; reconcile its manifest")
        if operation == "synthetic-auth":
            delivery = manifest["batches"]["1"]["deliveries"]["demo-2"]
            notebookutils.notebook.exit(json.dumps({"kind": kind, "ready": False,
                "run_id": auth.run_id, "batch_class": "bootstrap", "manifest_path": manifest_path,
                "batch_id": manifest["batches"]["1"]["batch_id"],
                "row_count": manifest["batches"]["1"]["row_count"],
                "sales_amount_minor_units": manifest["batches"]["1"]["sales_amount_minor_units"],
                "source_id": delivery["operation_id"], "ingestion_outcome": delivery["ingestion_outcome"],
                "verified_at_utc": delivery["verified_at_utc"], "status": "verified",
                "producer_job_id": None}, sort_keys=True))
        notebookutils.notebook.exit(json.dumps({"kind": kind, "ready": False,
            "run_id": run.run_id, "batch_class": "bootstrap", "manifest_path": manifest_path,
            "authorization_source_id": source_id,
            "batch_ids": [manifest["batches"][str(n)]["batch_id"] for n in (1, 2)],
            "status": "verified", "producer_job_id": None}, sort_keys=True))
elif operation in ("run-data", "run-native"):
    from fabric_runner import run_fabric_batches
    from fabric_runtime import RuntimeFabricClient, _uuid, build_adapters, native_parameters, prepare_measured, stop_requested
    from fabric_storage import OneLakeStore
    from generator import strict_json_loads
    from target_readback import OneLakeTableClient
    native = operation == "run-native"
    if native:
        document, selected, binding, probe_id, expiry = native_parameters(strict_json_loads(native_profile_json), now=datetime.now(timezone.utc))
    else:
        document = strict_json_loads(run_config_json)
        selected = strict_json_loads(selected_demos_json)
        binding = strict_json_loads(runtime_evidence_json)
    expected_identity = (_uuid(binding["tenant_id"]), _uuid(binding["identity_id"]))
    def checked_token(audience):
        token = notebookutils.credentials.getToken(audience)
        parts = token.split(".")
        if len(parts) != 3:
            raise RuntimeError("Runtime token has invalid identity claims")
        claims = strict_json_loads(base64.urlsafe_b64decode(parts[1] + "=" * (-len(parts[1]) % 4)).decode("utf-8"))
        if (_uuid(claims["tid"]), _uuid(claims["oid"])) != expected_identity:
            raise RuntimeError("Runtime token tenant or principal changed")
        return token
    audiences = {"pbi", "storage"} | ({"kusto"} if "demo-2" in selected else set())
    identities = {audience: expected_identity for audience in audiences if checked_token(audience)}
    fabric = RuntimeFabricClient(lambda: checked_token("pbi"))
    eventhouse = RuntimeFabricClient(lambda: checked_token("pbi"), skill="eventhouse-cli")
    clients = {name: eventhouse if name == "demo-2" else fabric for name in selected}
    tables = OneLakeTableClient(checked_token("storage")) if "demo-3" in selected else None
    context = notebookutils.runtime.context
    run, duration, targets, producer = prepare_measured(document, selected, probe_id, binding, context, identities, clients, tables)
    adapters = build_adapters(document, selected, notebookutils, token_provider=checked_token)
    store = OneLakeStore(producer["workspace_id"], producer["staging_lakehouse_id"], lambda: checked_token("storage"))
    try:
        if "demo-2" in adapters:
            from synthetic_fixture import peek_status_queues
            eventhouse_adapter = adapters["demo-2"]
            response = eventhouse_adapter.query_client.execute_query(eventhouse_adapter.database, "print ProducerRuntimeProbe=1")
            rows = list(response.primary_results[0])
            if len(rows) != 1 or rows[0]["ProducerRuntimeProbe"] != 1:
                raise RuntimeError("Runtime Eventhouse query returned unexpected results")
            peek_status_queues(eventhouse_adapter.ingest_client)
        if native:
            native_parameters(strict_json_loads(native_profile_json), now=datetime.now(timezone.utc))
        result = run_fabric_batches(store, run, adapters, duration, 1, selected_targets=targets,
                                    independent_source=True, fresh_only=native,
                                    demo3_admission=(expiry, binding["config_sha256"]) if native and "demo-3" in selected else None,
                                    stop_requested=lambda: stop_requested(store, run.run_id, probe_id, binding["config_sha256"]))
    finally:
        for adapter in adapters.values():
            for field in ("ingest_client", "query_client"):
                close = getattr(getattr(adapter, field, None), "close", None)
                if callable(close):
                    close()
    if not result["verified"]:
        raise RuntimeError("Producer delivery is not verified; reconcile its manifest")
    notebookutils.notebook.exit(json.dumps({"run_id": run.run_id, "status": "verified",
        "producer_job_id": None,
        "manifest_path": f"Files/producer/runs/{run.run_id}/manifest.json",
        "verification_scope": "file landing and Eventhouse ingestion, not downstream facts"}, sort_keys=True))
else:
    raise RuntimeError("Explicit supported operation required")
